In [1]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [4]:
userInputDataRaw = loadDataFromFile("User:Previous experiments")
timeSeriesData = loadDataFromFile("Data:Previous experiments")

/home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data
Loaded 498 records from /home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data/User:Previous experiments.json
/home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data
Loaded 697759 records from /home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data/Data:Previous experiments.json


In [5]:
userInputDataRaw

,_id,experimentState,timestamp,userInputCategory,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,pollutant-type,quantity-used,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,epoch_ms,timestamp_local
0,{'$oid': '6863fe54cdf92819382e6866'},StartingExperiment,{'$date': 1751383636513},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1751383636513,2025-07-01 15:27:16.513
1,{'$oid': '686408456a2505a88d9b0665'},InsertingSourcePollutant,{'$date': 1751386181232},ExperimentState,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,1751386181232,2025-07-01 16:09:41.232
2,{'$oid': '686409746a2505a88d9b087a'},RemovingSourcePollutant,{'$date': 1751386484742},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1751386484742,2025-07-01 16:14:44.742
3,{'$oid': '6865537ae30f2ab561c621a8'},StartingExperiment,{'$date': 1751470970240},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1751470970240,2025-07-02 15:42:50.240
4,{'$oid': '68655761e30f2ab561c629a2'},InsertingSourcePollutant,{'$date': 1751471969827},ExperimentState,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,1751471969827,2025-07-02 15:59:29.827
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493,{'$oid': '68ee725803b82ea0046b8c1c'},InsertingSourcePollutant,{'$date': 1760457304589},ExperimentState,None,None,None,1.50,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,1760457304589,2025-10-14 15:55:04.589
494,{'$oid': '68ee73a303b82ea0046b8ff8'},RemovingSourcePollutant,{'$date': 1760457635722},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1760457635722,2025-10-14 16:00:35.722
495,{'$oid': '68ee771f03b82ea0046b9a3e'},StartingExperiment,{'$date': 1760458527696},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1760458527696,2025-10-14 16:15:27.696
496,{'$oid': '68ee778703b82ea0046b9b76'},InsertingSourcePollutant,{'$date': 1760458631088},ExperimentState,None,None,None,0.50,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,1760458631088,2025-10-14 16:17:11.088


In [6]:
userInputDataRaw["room"].unique()

array([None, 'Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55',
       'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1',
       'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90',
       'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0',
       'Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 , Δ:2.00'],
      dtype=object)

In [7]:
timeSeriesData

,timestamp,Id=0:BME680:breathVocEquivalent,Id=1:BME680:breathVocEquivalent,Id=2:BME680:breathVocEquivalent
0,2025-07-01 15:27:18,NaN,1.480697,NaN
1,2025-07-01 15:27:19,NaN,NaN,1.430472
2,2025-07-01 15:27:21,NaN,1.469428,NaN
3,2025-07-01 15:27:22,NaN,NaN,1.441322
4,2025-07-01 15:27:24,NaN,1.476243,NaN
...,...,...,...,...
697754,2025-10-14 16:27:17,1.141398,NaN,NaN
697755,2025-10-14 16:27:17,NaN,1.138193,NaN
697756,2025-10-14 16:27:18,NaN,NaN,1.020477
697757,2025-10-14 16:27:20,1.137955,NaN,NaN


In [8]:
userInputData = userInputDataRaw.drop(columns=["_id","timestamp","epoch_ms","userInputCategory","pollutant-type","quantity-used"])

In [9]:
userInputData["timestamp_local"] = pd.to_datetime(userInputData["timestamp_local"])  # ensure datetime
userInputData["timestamp_local"] = userInputData["timestamp_local"].dt.floor("s")

In [10]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp_local
0,StartingExperiment,None,None,None,NaN,None,None,None,NaN,NaN,NaN,None,None,2025-07-01 15:27:16
1,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-01 16:09:41
2,RemovingSourcePollutant,None,None,None,NaN,None,None,None,NaN,NaN,NaN,None,None,2025-07-01 16:14:44
3,StartingExperiment,None,None,None,NaN,None,None,None,NaN,NaN,NaN,None,None,2025-07-02 15:42:50
4,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-02 15:59:29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493,InsertingSourcePollutant,None,None,None,1.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:55:04
494,RemovingSourcePollutant,None,None,None,NaN,None,None,None,NaN,NaN,NaN,None,None,2025-10-14 16:00:35
495,StartingExperiment,None,None,None,NaN,None,None,None,NaN,NaN,NaN,None,None,2025-10-14 16:15:27
496,InsertingSourcePollutant,None,None,None,0.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 16:17:11


In [11]:
userInputDataTemp = userInputData[userInputData["experimentState"].isin(["NoSourcePollutantInserted","InsertingSourcePollutant"])].copy()

userInputDataTemp["timestamp StartingExperiment"] = userInputData["timestamp_local"].shift(1).loc[userInputData.index]
userInputDataTemp["timestamp EndingExperiment"]= userInputData["timestamp_local"].shift(-1).loc[userInputData.index]
userInputData =userInputDataTemp
userInputData = userInputData.reset_index(drop = True)
userInputData = userInputData.rename(columns={"timestamp_local":"timestamp InsertingSource"})



In [12]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment
0,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44
1,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52
2,InsertingSourcePollutant,on,None,None,0.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31
3,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.85,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11
4,InsertingSourcePollutant,on,None,None,1.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.35,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49
162,InsertingSourcePollutant,None,None,None,1.80,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.50,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46
163,InsertingSourcePollutant,None,None,None,2.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21
164,InsertingSourcePollutant,None,None,None,1.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:55:04,2025-10-14 15:53:21,2025-10-14 16:00:35


In [13]:
userInputData["room"].unique()

array(['Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55',
       'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1',
       'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90',
       'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0',
       'Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 , Δ:2.00'],
      dtype=object)

In [14]:
dict_of_timeseries = {}
for index, row in userInputData.iterrows():
    #get datetime
    start = row["timestamp StartingExperiment"]
    turning_point = row["timestamp InsertingSource"]
    end = row["timestamp EndingExperiment"]
    #Grab data
    particular_experiment = timeSeriesData[(timeSeriesData["timestamp"] >= start) & (timeSeriesData["timestamp"] <= end)]
    particular_experiment = particular_experiment.sort_values(by = ["timestamp"],ignore_index = True)
    dict_of_timeseries[index] = particular_experiment



In [15]:
example_df = dict_of_timeseries[10]
example_df

,timestamp,Id=0:BME680:breathVocEquivalent,Id=1:BME680:breathVocEquivalent,Id=2:BME680:breathVocEquivalent
0,2025-07-19 16:04:22,NaN,0.513479,NaN
1,2025-07-19 16:04:24,NaN,NaN,0.508068
2,2025-07-19 16:04:25,NaN,0.518917,NaN
3,2025-07-19 16:04:27,NaN,NaN,0.504020
4,2025-07-19 16:04:28,NaN,0.521494,NaN
...,...,...,...,...
699,2025-07-19 16:21:49,NaN,2.617029,NaN
700,2025-07-19 16:21:51,NaN,NaN,5.626779
701,2025-07-19 16:21:52,NaN,2.615562,NaN
702,2025-07-19 16:21:54,NaN,NaN,5.601505


In [16]:
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index]

    #make the dataframe from multiple columns to have the data for each sensor to just one column
    particular_experiment = particular_experiment.melt(
        id_vars=["timestamp"], 
        value_vars=[
            "Id=0:BME680:breathVocEquivalent",
            "Id=1:BME680:breathVocEquivalent",
            "Id=2:BME680:breathVocEquivalent"
        ],
        var_name="sensors", 
        value_name="VOC"
    )
    particular_experiment = particular_experiment.dropna(ignore_index = True)

    particular_experiment = particular_experiment.sort_values(by = ["timestamp"],ignore_index = True)
    dict_of_timeseries[index] = particular_experiment

In [17]:
example_df = dict_of_timeseries[10]
example_df.head(20)

,timestamp,sensors,VOC
0,2025-07-19 16:04:22,Id=1:BME680:breathVocEquivalent,0.513479
1,2025-07-19 16:04:24,Id=2:BME680:breathVocEquivalent,0.508068
2,2025-07-19 16:04:25,Id=1:BME680:breathVocEquivalent,0.518917
3,2025-07-19 16:04:27,Id=2:BME680:breathVocEquivalent,0.504020
4,2025-07-19 16:04:28,Id=1:BME680:breathVocEquivalent,0.521494
5,2025-07-19 16:04:30,Id=2:BME680:breathVocEquivalent,0.509921
6,2025-07-19 16:04:31,Id=1:BME680:breathVocEquivalent,0.518891
7,2025-07-19 16:04:33,Id=2:BME680:breathVocEquivalent,0.507465
8,2025-07-19 16:04:34,Id=1:BME680:breathVocEquivalent,0.514929
9,2025-07-19 16:04:36,Id=2:BME680:breathVocEquivalent,0.509477


In [18]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment
0,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44
1,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52
2,InsertingSourcePollutant,on,None,None,0.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31
3,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.85,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11
4,InsertingSourcePollutant,on,None,None,1.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.35,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49
162,InsertingSourcePollutant,None,None,None,1.80,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.50,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46
163,InsertingSourcePollutant,None,None,None,2.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21
164,InsertingSourcePollutant,None,None,None,1.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:55:04,2025-10-14 15:53:21,2025-10-14 16:00:35


In [19]:
for index, row in userInputData.iterrows():
#Now, we only have timestamps based of whenever the sensor took the value, but we want to have times 
    # Get full range of seconds from start to end timestamp
    particular_experiment =  dict_of_timeseries[index]
    start = row["timestamp StartingExperiment"]
    turning_point = row["timestamp InsertingSource"]

    end = row["timestamp EndingExperiment"]
    full_time_range = pd.date_range(start, end, freq="s")
    
    all_sensors = particular_experiment["sensors"].unique()
    particular_experiment["timestamp"] = pd.to_datetime(particular_experiment["timestamp"])
    
    full_index = pd.MultiIndex.from_product([full_time_range, all_sensors], names=["timestamp", "sensors"])
    extended_particular_experiment= pd.DataFrame(index=full_index).reset_index()
    particular_experiment = extended_particular_experiment.merge(particular_experiment, on=["timestamp", "sensors"], how="left",suffixes=["",None])
    particular_experiment["after_insertion"] = particular_experiment["timestamp"] > turning_point
    particular_experiment["original_value"] = pd.notna(particular_experiment["VOC"])
    dict_of_timeseries[index] = particular_experiment


In [20]:
userInputData["date of experiment"] = userInputData["timestamp EndingExperiment"].apply(lambda x:x.date())

In [21]:
example_df = dict_of_timeseries[10]
example_df

,timestamp,sensors,VOC,after_insertion,original_value
0,2025-07-19 16:04:22,Id=1:BME680:breathVocEquivalent,0.513479,False,True
1,2025-07-19 16:04:22,Id=2:BME680:breathVocEquivalent,NaN,False,False
2,2025-07-19 16:04:23,Id=1:BME680:breathVocEquivalent,NaN,False,False
3,2025-07-19 16:04:23,Id=2:BME680:breathVocEquivalent,NaN,False,False
4,2025-07-19 16:04:24,Id=1:BME680:breathVocEquivalent,NaN,False,False
...,...,...,...,...,...
2104,2025-07-19 16:21:53,Id=2:BME680:breathVocEquivalent,NaN,True,False
2105,2025-07-19 16:21:54,Id=1:BME680:breathVocEquivalent,NaN,True,False
2106,2025-07-19 16:21:54,Id=2:BME680:breathVocEquivalent,5.601505,True,True
2107,2025-07-19 16:21:55,Id=1:BME680:breathVocEquivalent,2.590167,True,True


In [22]:
for index, row in userInputData.iterrows():
    
        particular_experiment =  dict_of_timeseries[index]

        particular_experiment["VOC"] = (
            particular_experiment
            .groupby("sensors")["VOC"]
            .transform(lambda s: s.interpolate(method = "cubic"))
        )
        dict_of_timeseries[index] = particular_experiment

In [23]:
example_df = dict_of_timeseries[10]
example_df

,timestamp,sensors,VOC,after_insertion,original_value
0,2025-07-19 16:04:22,Id=1:BME680:breathVocEquivalent,0.513479,False,True
1,2025-07-19 16:04:22,Id=2:BME680:breathVocEquivalent,NaN,False,False
2,2025-07-19 16:04:23,Id=1:BME680:breathVocEquivalent,0.515345,False,False
3,2025-07-19 16:04:23,Id=2:BME680:breathVocEquivalent,NaN,False,False
4,2025-07-19 16:04:24,Id=1:BME680:breathVocEquivalent,0.517211,False,False
...,...,...,...,...,...
2104,2025-07-19 16:21:53,Id=2:BME680:breathVocEquivalent,5.629334,True,False
2105,2025-07-19 16:21:54,Id=1:BME680:breathVocEquivalent,2.601317,True,False
2106,2025-07-19 16:21:54,Id=2:BME680:breathVocEquivalent,5.601505,True,True
2107,2025-07-19 16:21:55,Id=1:BME680:breathVocEquivalent,2.590167,True,True


In [24]:
# Ensure the columns exist and have datetime dtype
userInputData["actual timestamp StartingExperiment"] = pd.NaT
userInputData["actual timestamp EndingExperiment"] = pd.NaT
userInputData = userInputData.astype({
    "actual timestamp StartingExperiment": "datetime64[ns]",
    "actual timestamp EndingExperiment": "datetime64[ns]"
})

for index, row in userInputData.iterrows():
    particular_experiment = dict_of_timeseries[index]

    first_timestamps = particular_experiment[particular_experiment["VOC"].notna()] \
        .groupby("sensors", as_index=False)["timestamp"].min()
    last_timestamps = particular_experiment[particular_experiment["VOC"].notna()] \
        .groupby("sensors", as_index=False)["timestamp"].max()

    start = first_timestamps["timestamp"].max()
    end = last_timestamps["timestamp"].min()

    userInputData.at[index, "actual timestamp StartingExperiment"] = start
    userInputData.at[index, "actual timestamp EndingExperiment"] = end

    particular_experiment = particular_experiment.loc[
        particular_experiment["timestamp"].between(start, end, inclusive="both")
    ].reset_index(drop=True)

    dict_of_timeseries[index] = particular_experiment


In [25]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment
0,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42
1,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50
2,InsertingSourcePollutant,on,None,None,0.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31
3,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.85,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10
4,InsertingSourcePollutant,on,None,None,1.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.35,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47
162,InsertingSourcePollutant,None,None,None,1.80,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.50,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44
163,InsertingSourcePollutant,None,None,None,2.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20
164,InsertingSourcePollutant,None,None,None,1.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:55:04,2025-10-14 15:53:21,2025-10-14 16:00:35,2025-10-14,2025-10-14 15:53:23,2025-10-14 16:00:33


In [26]:
example_df = dict_of_timeseries[10]
example_df

,timestamp,sensors,VOC,after_insertion,original_value
0,2025-07-19 16:04:24,Id=1:BME680:breathVocEquivalent,0.517211,False,False
1,2025-07-19 16:04:24,Id=2:BME680:breathVocEquivalent,0.508068,False,True
2,2025-07-19 16:04:25,Id=1:BME680:breathVocEquivalent,0.518917,False,True
3,2025-07-19 16:04:25,Id=2:BME680:breathVocEquivalent,0.503896,False,False
4,2025-07-19 16:04:26,Id=1:BME680:breathVocEquivalent,0.520305,False,False
...,...,...,...,...,...
2098,2025-07-19 16:21:52,Id=2:BME680:breathVocEquivalent,5.635052,True,False
2099,2025-07-19 16:21:53,Id=1:BME680:breathVocEquivalent,2.609777,True,False
2100,2025-07-19 16:21:53,Id=2:BME680:breathVocEquivalent,5.629334,True,False
2101,2025-07-19 16:21:54,Id=1:BME680:breathVocEquivalent,2.601317,True,False


In [27]:


for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index]

    print(f"index is {index}")
    particular_experiment =  dict_of_timeseries[index]
    
    # keep only rows with VOC present
    filtered = particular_experiment[particular_experiment["VOC"].notna()]
    min_timestamp_for_each_sensor = (
        filtered
        .groupby("sensors", as_index=False)["timestamp"]
        .min()
        )
    print(f"min value")
    print(min_timestamp_for_each_sensor)
    first_timestamp_to_keep = min_timestamp_for_each_sensor["timestamp"].max()
    
    particular_experiment = particular_experiment[particular_experiment["timestamp"] >= first_timestamp_to_keep]
    
    print(f"max value")
    max_timestamp_for_each_sensor =(
        filtered
        .groupby("sensors", as_index=False)["timestamp"]
        .max()
        )
    print(max_timestamp_for_each_sensor)
 

index is 0
min value
                           sensors           timestamp
0  Id=1:BME680:breathVocEquivalent 2025-07-01 15:27:19
1  Id=2:BME680:breathVocEquivalent 2025-07-01 15:27:19
max value
                           sensors           timestamp
0  Id=1:BME680:breathVocEquivalent 2025-07-01 16:14:42
1  Id=2:BME680:breathVocEquivalent 2025-07-01 16:14:42
index is 1
min value
                           sensors           timestamp
0  Id=1:BME680:breathVocEquivalent 2025-07-02 15:42:50
1  Id=2:BME680:breathVocEquivalent 2025-07-02 15:42:50
max value
                           sensors           timestamp
0  Id=1:BME680:breathVocEquivalent 2025-07-02 16:04:50
1  Id=2:BME680:breathVocEquivalent 2025-07-02 16:04:50
index is 2
min value
                           sensors           timestamp
0  Id=1:BME680:breathVocEquivalent 2025-07-03 12:30:25
1  Id=2:BME680:breathVocEquivalent 2025-07-03 12:30:25
max value
                           sensors           timestamp
0  Id=1:BME680:breathVocEqu

In [28]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment
0,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42
1,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50
2,InsertingSourcePollutant,on,None,None,0.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31
3,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.85,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10
4,InsertingSourcePollutant,on,None,None,1.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.35,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47
162,InsertingSourcePollutant,None,None,None,1.80,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.50,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44
163,InsertingSourcePollutant,None,None,None,2.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20
164,InsertingSourcePollutant,None,None,None,1.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:55:04,2025-10-14 15:53:21,2025-10-14 16:00:35,2025-10-14,2025-10-14 15:53:23,2025-10-14 16:00:33


In [29]:
index = 6
particular_experiment =  dict_of_timeseries[index]
sensors = particular_experiment["sensors"].unique()

print(index)
for sensor in sensors:
    mask = particular_experiment["sensors"]==sensor
    masked_data = particular_experiment.loc[mask]
    diffs = masked_data["timestamp"].diff()
    expected_step = diffs.mode()[0]
    broken_rows = masked_data["timestamp"][diffs != expected_step]
    print("Rows breaking frequency:\n", broken_rows)
    print()

6
Rows breaking frequency:
 0      2025-07-14 18:30:00
1177   2025-07-14 18:39:48
2372   2025-07-14 18:49:45
5094   2025-07-14 19:12:24
Name: timestamp, dtype: datetime64[ns]

Rows breaking frequency:
 1      2025-07-14 18:30:00
2500   2025-07-14 18:50:48
4229   2025-07-14 19:05:12
4608   2025-07-14 19:08:21
Name: timestamp, dtype: datetime64[ns]



In [30]:
for index, row in userInputData.iterrows():

    particular_experiment =  dict_of_timeseries[index]
    particular_experiment = particular_experiment.drop_duplicates()
    dict_of_timeseries[index] = particular_experiment

In [31]:
dict_of_timeseries[7]

,timestamp,sensors,VOC,after_insertion,original_value
0,2025-07-19 14:59:49,Id=2:BME680:breathVocEquivalent,0.680033,False,False
1,2025-07-19 14:59:49,Id=1:BME680:breathVocEquivalent,2.118851,False,True
2,2025-07-19 14:59:50,Id=2:BME680:breathVocEquivalent,0.679083,False,False
3,2025-07-19 14:59:50,Id=1:BME680:breathVocEquivalent,2.125827,False,False
4,2025-07-19 14:59:51,Id=2:BME680:breathVocEquivalent,0.681077,False,True
...,...,...,...,...,...
997,2025-07-19 15:08:07,Id=1:BME680:breathVocEquivalent,1.193376,True,True
998,2025-07-19 15:08:08,Id=2:BME680:breathVocEquivalent,1.186056,True,False
999,2025-07-19 15:08:08,Id=1:BME680:breathVocEquivalent,1.192881,True,False
1000,2025-07-19 15:08:09,Id=2:BME680:breathVocEquivalent,1.184700,True,True


In [32]:
userInputData["time taken total"] = np.nan

for index, row in userInputData.iterrows():

    particular_experiment =  dict_of_timeseries[index]
    sensors = particular_experiment["sensors"].unique()
    particular_experiment["datetime_timestamp"] = particular_experiment["timestamp"]
    particular_experiment = particular_experiment.drop(columns="timestamp")
    particular_experiment["timestamp"] = np.nan
    particular_experiment["timestamp"] = particular_experiment["timestamp"].astype("timedelta64[ns]")
    for sensor in sensors:
        
        mask = particular_experiment["sensors"]==sensor
        #masked_data = particular_experiment.loc[mask]
        
        start = particular_experiment.loc[mask]["datetime_timestamp"].min()
        end = particular_experiment.loc[mask]["datetime_timestamp"].max()
        
        duration = end - start
        
        timeline = pd.timedelta_range(start="0s", end=duration, freq="s")
        
        particular_experiment.loc[mask,"timestamp"] = timeline
        particular_experiment.loc[mask,"seconds"] = timeline.total_seconds()
        userInputData.at[index,"time taken total"] = duration    
    dict_of_timeseries[index] =   particular_experiment
       


/tmp/ipykernel_6685/3649789654.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0 days 00:47:23' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  userInputData.at[index,"time taken total"] = duration
/tmp/ipykernel_6685/3649789654.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  particular_experiment["datetime_timestamp"] = particular_experiment["timestamp"]


In [33]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total
0,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23
1,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00
2,InsertingSourcePollutant,on,None,None,0.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06
3,InsertingSourcePollutant,on,None,None,0.95,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.85,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30
4,InsertingSourcePollutant,on,None,None,1.40,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.35,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47,0 days 00:08:27
162,InsertingSourcePollutant,None,None,None,1.80,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.50,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44,0 days 00:09:27
163,InsertingSourcePollutant,None,None,None,2.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20,0 days 00:07:27
164,InsertingSourcePollutant,None,None,None,1.50,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.30,None,None,2025-10-14 15:55:04,2025-10-14 15:53:21,2025-10-14 16:00:35,2025-10-14,2025-10-14 15:53:23,2025-10-14 16:00:33,0 days 00:07:10


In [34]:
 dict_of_timeseries[10]

,sensors,VOC,after_insertion,original_value,datetime_timestamp,timestamp,seconds
0,Id=1:BME680:breathVocEquivalent,0.517211,False,False,2025-07-19 16:04:24,0 days 00:00:00,0.0
1,Id=2:BME680:breathVocEquivalent,0.508068,False,True,2025-07-19 16:04:24,0 days 00:00:00,0.0
2,Id=1:BME680:breathVocEquivalent,0.518917,False,True,2025-07-19 16:04:25,0 days 00:00:01,1.0
3,Id=2:BME680:breathVocEquivalent,0.503896,False,False,2025-07-19 16:04:25,0 days 00:00:01,1.0
4,Id=1:BME680:breathVocEquivalent,0.520305,False,False,2025-07-19 16:04:26,0 days 00:00:02,2.0
...,...,...,...,...,...,...,...
2098,Id=2:BME680:breathVocEquivalent,5.635052,True,False,2025-07-19 16:21:52,0 days 00:17:28,1048.0
2099,Id=1:BME680:breathVocEquivalent,2.609777,True,False,2025-07-19 16:21:53,0 days 00:17:29,1049.0
2100,Id=2:BME680:breathVocEquivalent,5.629334,True,False,2025-07-19 16:21:53,0 days 00:17:29,1049.0
2101,Id=1:BME680:breathVocEquivalent,2.601317,True,False,2025-07-19 16:21:54,0 days 00:17:30,1050.0


In [35]:
userInputData["timestamp InsertingSource timedelta"] = np.nan
userInputData["timestamp InsertingSource seconds"] = np.nan

for index, row in userInputData.iterrows():

    particular_experiment =  dict_of_timeseries[index]
    sensors = particular_experiment["sensors"].unique()

    for sensor in sensors:
        
        mask = particular_experiment["sensors"]==sensor
        #print(masker)

        start = particular_experiment.loc[mask,:]["timestamp"].min()
       # print(start)
        end = particular_experiment.loc[mask,:]["timestamp"].max()
        #print(end)
       # print(sensor)
        
        mask = (particular_experiment["sensors"] == sensor) & (particular_experiment["after_insertion"] == True)
        filtered = particular_experiment.loc[mask]
        #print(first_index)
        if not filtered.empty:
            first_index = filtered.index[0]
            insertion_time = filtered.at[first_index, "timestamp"]
        
            userInputData.at[index, "timestamp InsertingSource timedelta"] = insertion_time
            userInputData.at[index, "timestamp InsertingSource seconds"] = insertion_time.total_seconds()
            userInputData.at[index, "time taken after insertion"] = (
                userInputData.at[index, "time taken total"] - insertion_time
    )

    dict_of_timeseries[index] =   particular_experiment
    

/tmp/ipykernel_6685/161819636.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0 days 00:42:23' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  userInputData.at[index, "timestamp InsertingSource timedelta"] = insertion_time


In [36]:
from sklearn.preprocessing import StandardScaler

#use rolling window
for index, row in userInputData.iterrows():
    particular_experiment =  dict_of_timeseries[index].copy()
    sensors = particular_experiment["sensors"].unique()

    for sensor in sensors:
        mask = particular_experiment["sensors"]==sensor
        particular_experiment.loc[mask,"VOC rolling average"] = particular_experiment.loc[mask].rolling(30,center= True ,min_periods=1,on="seconds")["VOC"].mean()

        particular_experiment.loc[mask,"VOC gradient"] = np.gradient(particular_experiment.loc[mask,"VOC rolling average"].to_numpy())
        particular_experiment.loc[mask,"VOC gradient cumsum"] = particular_experiment.loc[mask,"VOC gradient"].cumsum()
        
    scaler = StandardScaler()
    particular_experiment.loc[:,"VOC standard scaled"] = scaler.fit_transform(particular_experiment.loc[:,"VOC"].to_numpy().reshape(-1,1))
    particular_experiment.loc[:,"VOC rolling average standard scaled"] = scaler.fit_transform(particular_experiment.loc[:,"VOC rolling average"].to_numpy().reshape(-1,1))


    dict_of_timeseries[index] = particular_experiment  

In [37]:
dict_of_timeseries[6]

,sensors,VOC,after_insertion,original_value,datetime_timestamp,timestamp,seconds,VOC rolling average,VOC gradient,VOC gradient cumsum,VOC standard scaled,VOC rolling average standard scaled
0,Id=1:BME680:breathVocEquivalent,0.500000,False,True,2025-07-14 18:30:00,0 days 00:00:00,0.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
1,Id=2:BME680:breathVocEquivalent,0.500000,False,True,2025-07-14 18:30:00,0 days 00:00:00,0.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
2,Id=1:BME680:breathVocEquivalent,0.500000,False,False,2025-07-14 18:30:01,0 days 00:00:01,1.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
3,Id=2:BME680:breathVocEquivalent,0.500000,False,False,2025-07-14 18:30:01,0 days 00:00:01,1.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
4,Id=1:BME680:breathVocEquivalent,0.500000,False,False,2025-07-14 18:30:02,0 days 00:00:02,2.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
...,...,...,...,...,...,...,...,...,...,...,...,...
6009,Id=2:BME680:breathVocEquivalent,18.846076,False,False,2025-07-14 19:20:01,0 days 00:50:01,3001.0,19.710940,-0.070793,19.176005,-0.745307,-0.746149
6010,Id=1:BME680:breathVocEquivalent,45.835785,False,False,2025-07-14 19:20:02,0 days 00:50:02,3002.0,48.097630,-0.234154,47.481665,-0.688386,-0.686067
6011,Id=2:BME680:breathVocEquivalent,18.729567,False,False,2025-07-14 19:20:02,0 days 00:50:02,3002.0,19.641070,-0.068507,19.107498,-0.745552,-0.746297
6012,Id=1:BME680:breathVocEquivalent,45.545510,False,True,2025-07-14 19:20:03,0 days 00:50:03,3003.0,47.865699,-0.231930,47.249734,-0.688998,-0.686557


In [38]:
userInputData.loc[:,["front-wall","side-right-wall","back-wall","side-left-wall"]] = np.floor(userInputData.loc[:,["front-wall","side-right-wall","back-wall","side-left-wall"]]*10)/10

In [39]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion
0,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543.0,0 days 00:05:00
1,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000.0,0 days 00:05:20
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248.0,0 days 00:10:58
3,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88.0,0 days 00:06:02
4,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.3,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05,0 days 00:17:36,1056.0,0 days 00:08:29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47,0 days 00:08:27,0 days 00:00:54,54.0,0 days 00:07:33
162,InsertingSourcePollutant,None,None,None,1.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.5,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44,0 days 00:09:27,0 days 00:00:40,40.0,0 days 00:08:47
163,InsertingSourcePollutant,None,None,None,2.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20,0 days 00:07:27,0 days 00:01:35,95.0,0 days 00:05:52
164,InsertingSourcePollutant,None,None,None,1.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:55:04,2025-10-14 15:53:21,2025-10-14 16:00:35,2025-10-14,2025-10-14 15:53:23,2025-10-14 16:00:33,0 days 00:07:10,0 days 00:01:42,102.0,0 days 00:05:28


In [40]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion
0,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543.0,0 days 00:05:00
1,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000.0,0 days 00:05:20
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248.0,0 days 00:10:58
3,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88.0,0 days 00:06:02
4,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.3,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05,0 days 00:17:36,1056.0,0 days 00:08:29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47,0 days 00:08:27,0 days 00:00:54,54.0,0 days 00:07:33
162,InsertingSourcePollutant,None,None,None,1.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.5,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44,0 days 00:09:27,0 days 00:00:40,40.0,0 days 00:08:47
163,InsertingSourcePollutant,None,None,None,2.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20,0 days 00:07:27,0 days 00:01:35,95.0,0 days 00:05:52
164,InsertingSourcePollutant,None,None,None,1.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:55:04,2025-10-14 15:53:21,2025-10-14 16:00:35,2025-10-14,2025-10-14 15:53:23,2025-10-14 16:00:33,0 days 00:07:10,0 days 00:01:42,102.0,0 days 00:05:28


In [41]:
room_length = 4.0
room_width = 3.25
userInputData["x axis"] = np.nan
userInputData["y axis"] = np.nan
#front is max,back is zero,right is max,left is zero
def find_x_axis(x,room_width):
    from_right = None
    from_left = None

    #return NaN either if numbers at the position columns are negative or above room width
    if (x["side-right-wall"]<0 and x["side-right-wall"]> room_width) or (x["side-left-wall"]< 0 and x["side-left-wall"]> room_width) or (pd.isna(x["side-right-wall"]) and pd.isna(x["side-left-wall"])):
        x["x axis"] = np.nan
        return x["x axis"] 
    if pd.notna(x["side-right-wall"]):
       from_right = -x["side-right-wall"]
    if pd.notna(x["side-left-wall"]):
       from_left = -room_width + x["side-left-wall"]
      
    if  from_right is not None and from_left is not None:
        if from_right == from_left:
            x["x axis"] = from_right
        else:
            x["x axis"] = np.nan
          

    elif pd.notna(x["side-left-wall"]):
    
        x["x axis"] = from_left

    elif pd.notna(x["side-right-wall"]):
        x["x axis"] = from_right
    return  x["x axis"]

def find_y_axis(x,room_length):
    from_front = None
    from_back = None

    #return NaN either if numbers at the position columns are negative or above room width
    if (x["front-wall"]<0 and x["front-wall"]> room_length) or (x["back-wall"]< 0 and x["back-wall"]> room_length) or (pd.isna(x["front-wall"]) and pd.isna(x["back-wall"])):
        x["y axis"] = np.nan
        return x["y axis"] 
    if pd.notna(x["front-wall"]):
       from_front = room_length - x["front-wall"]
    if pd.notna(x["back-wall"]):
       from_back = x["back-wall"]
    if from_front is not None and from_back is not None:
        if from_front == from_back:
            x["y axis"] = from_front
        else:
            x["y axis"] = np.nan
            
    elif pd.notna(x["back-wall"]):
    
        x["y axis"] = from_back

    elif pd.notna(x["front-wall"]):
        x["y axis"] = from_front

    return  x["y axis"]

userInputData["x axis"] = userInputData.apply(lambda x:find_x_axis(x,room_width),axis=1)
userInputData["y axis"] = userInputData.apply(lambda x:find_y_axis(x,room_length),axis=1)

In [42]:
userInputData.head(30)

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,x axis,y axis
0,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543.0,0 days 00:05:00,-2.35,3.1
1,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000.0,0 days 00:05:20,-2.35,3.1
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248.0,0 days 00:10:58,-2.35,3.6
3,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88.0,0 days 00:06:02,-1.45,3.1
4,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.3,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05,0 days 00:17:36,1056.0,0 days 00:08:29,-1.95,2.6
5,InsertingSourcePollutant,None,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.7,None,None,2025-07-12 22:57:57,2025-07-12 22:31:25,2025-07-12 23:02:13,2025-07-12,2025-07-12 22:34:03,2025-07-12 23:02:11,0 days 00:28:08,0 days 00:23:55,1435.0,0 days 00:04:13,-1.55,3.1
6,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.7,None,None,2025-07-14 19:40:37,2025-07-14 17:50:04,2025-07-14 19:48:32,2025-07-14,2025-07-14 18:30:00,2025-07-14 19:20:03,0 days 00:50:03,NaN,NaN,NaT,-1.55,2.6
7,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:04:09,2025-07-19 14:59:47,2025-07-19 15:08:10,2025-07-19,2025-07-19 14:59:49,2025-07-19 15:08:09,0 days 00:08:20,0 days 00:04:21,261.0,0 days 00:03:59,-2.15,2.2
8,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:17:10,2025-07-19 15:14:13,2025-07-19 15:21:23,2025-07-19,2025-07-19 15:14:15,2025-07-19 15:21:21,0 days 00:07:06,0 days 00:02:56,176.0,0 days 00:04:10,-2.15,2.2
9,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:30:49,2025-07-19 15:25:07,2025-07-19 15:44:28,2025-07-19,2025-07-19 15:25:09,2025-07-19 15:44:27,0 days 00:19:18,0 days 00:05:41,341.0,0 days 00:13:37,-2.15,2.2


In [43]:
userInputData.loc[:, ["x axis", "y axis"]].tail(30)

,x axis,y axis
136,-1.50,2.5
137,-1.50,3.5
138,-2.50,3.5
139,NaN,NaN
140,-2.50,2.5
141,-2.50,1.5
142,-2.50,0.5
143,-2.95,0.5
144,-2.95,1.5
145,NaN,NaN


In [44]:
userInputData["room"].unique()

array(['Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55',
       'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1',
       'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90',
       'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0',
       'Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 , Δ:2.00'],
      dtype=object)

In [45]:
room_length = 4.0
room_width = 3.25
pos_id0 = 'position of Id=0:BME680:breathVocEquivalent'
pos_id1 = 'position of Id=1:BME680:breathVocEquivalent'
pos_id2 = 'position of Id=2:BME680:breathVocEquivalent'
pos_id_all = [pos_id0,pos_id1,pos_id2]
room_positions = {
    'Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55' : {
        pos_id1 +"-x axis" : room_width - 0.85 ,
        pos_id1 +"-y axis" : room_length - 0.95,
        pos_id2+"-x axis" :  room_width - 1.55 ,
        pos_id2+  "-y axis" : room_length - 0.95
              
    },
        
    'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1' : {
        pos_id1+"-x axis" :  room_width - 1.2 ,
        pos_id1+"-y axis" :  0.6,
        pos_id2+"-x axis" :  room_width -1.1 ,
        pos_id2+"-y axis" : room_length - 0.8
   
    },      
        
    'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90' : {
        
        pos_id0+"-x axis" : -room_width + 0.9, 
        pos_id0+"-y axis" : room_length - 0.8,
        
        pos_id1+"-x axis" : -room_width + 0.9,
        pos_id1+"-y axis" : room_length - 0.8,
        
        pos_id2+"-x axis" : -room_width + 0.9,
        pos_id2+"-y axis" : room_length - 0.8
     },
    'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0' :{
        pos_id0+"-x axis" : -room_width  + 1.4, 
        pos_id0+"-y axis" : room_length - 0.5,
        
        pos_id1+"-x axis" :   -0.5,
        pos_id1+"-y axis" :   1.7,
        
        pos_id2+"-x axis" :  -room_width +1.0,
        pos_id2+"-y axis" :  0.4    
    }
}
room_positions

{'Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55': {'position of Id=1:BME680:breathVocEquivalent-x axis': 2.4,
  'position of Id=1:BME680:breathVocEquivalent-y axis': 3.05,
  'position of Id=2:BME680:breathVocEquivalent-x axis': 1.7,
  'position of Id=2:BME680:breathVocEquivalent-y axis': 3.05},
 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1': {'position of Id=1:BME680:breathVocEquivalent-x axis': 2.05,
  'position of Id=1:BME680:breathVocEquivalent-y axis': 0.6,
  'position of Id=2:BME680:breathVocEquivalent-x axis': 2.15,
  'position of Id=2:BME680:breathVocEquivalent-y axis': 3.2},
 'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90': {'position of Id=0:BME680:breathVocEquivalent-x axis': -2.35,
  'position of Id=0:BME680:breathVocEquivalent-y axis': 3.2,
  'position of Id=1:BME680:breathVocEquivalent-x axis': -2.35,
  'position of Id=1:BME680:breathVocEquivalent-y axis': 3.2,
  'position of Id=2:BME680:breathVocEquivalent-x axis': -2.35,
  'position of Id=2:BME680:breathVocEqui

In [46]:
pos_id0 = 'position of Id=0:BME680:breathVocEquivalent'
pos_id1 = 'position of Id=1:BME680:breathVocEquivalent'
pos_id2 = 'position of Id=2:BME680:breathVocEquivalent'
pos_id_all = [pos_id0,pos_id1,pos_id2]
for pos_id in pos_id_all:
    userInputData[pos_id+"-x axis"] = np.nan
    userInputData[pos_id+"-y axis"] = np.nan
    


for room,sensor_positions in room_positions.items():
   # print(f"room:{room}")
    outer_mask = userInputData["room"] == room
  #  print(f"sensor_positions:{sensor_positions}")
    for sensor_position,value  in sensor_positions.items():
 
        userInputData.loc[outer_mask,sensor_position] = value

In [47]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,x axis,y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis
0,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543.0,0 days 00:05:00,-2.35,3.1,NaN,NaN,2.4,3.05,1.70,3.05
1,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000.0,0 days 00:05:20,-2.35,3.1,NaN,NaN,2.4,3.05,1.70,3.05
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248.0,0 days 00:10:58,-2.35,3.6,NaN,NaN,2.4,3.05,1.70,3.05
3,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88.0,0 days 00:06:02,-1.45,3.1,NaN,NaN,2.4,3.05,1.70,3.05
4,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.3,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05,0 days 00:17:36,1056.0,0 days 00:08:29,-1.95,2.6,NaN,NaN,2.4,3.05,1.70,3.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47,0 days 00:08:27,0 days 00:00:54,54.0,0 days 00:07:33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,InsertingSourcePollutant,None,None,None,1.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.5,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44,0 days 00:09:27,0 days 00:00:40,40.0,0 days 00:08:47,-2.75,2.2,NaN,NaN,NaN,NaN,NaN,NaN
163,InsertingSourcePollutant,None,None,None,2.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20,0 days 00:07:27,0 days 00:01:35,95.0,0 days 00:05:52,-2.95,1.5,-1.85,3.5,-0.5,1.70,-2.25,0.40
164,InsertingSourcePollutant,None,None,None,1.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:55:04,2025-10-14 15:53:21,2025-10-14 16:00:35,2025-10-14,2025-10-14 15:53:23,2025-10-14 16:00:33,0 days 00:07:10,0 days 00:01:42,102.0,0 days 00:05:28,-2.95,2.5,-1.85,3.5,-0.5,1.70,-2.25

In [48]:
#find euclidian distance between sensors position and source

x_axis_source = userInputData["x axis"].to_numpy()
y_axis_source = userInputData["y axis"].to_numpy()


for id_sensor_number in range(3):
    sensor_name = "Id="+str(id_sensor_number)+":BME680:breathVocEquivalent"
    x_axis_sensor = userInputData["position of "+ sensor_name + "-x axis"].to_numpy()
    y_axis_sensor = userInputData["position of "+ sensor_name + "-y axis"].to_numpy()
    # Euclidean distance per row
    distances = np.sqrt((x_axis_sensor - x_axis_source)**2 + (y_axis_sensor - y_axis_source)**2)
    var_name = "Euclidian distance to "+sensor_name
    userInputData[var_name] = distances


In [49]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,x axis,y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis,Euclidian distance to Id=0:BME680:breathVocEquivalent,Euclidian distance to Id=1:BME680:breathVocEquivalent,Euclidian distance to Id=2:BME680:breathVocEquivalent
0,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543.0,0 days 00:05:00,-2.35,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.750263,4.050309
1,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000.0,0 days 00:05:20,-2.35,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.750263,4.050309
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248.0,0 days 00:10:58,-2.35,3.6,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.781736,4.087175
3,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88.0,0 days 00:06:02,-1.45,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,3.850325,3.150397
4,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.3,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05,0 days 00:17:36,1056.0,0 days 00:08:29,-1.95,2.6,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.373214,3.677635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47,0 days 00:08:27,0 days 00:00:54,54.0,0 days 00:07:33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,InsertingSourcePollutant,None,None,None,1.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.5,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44,0 days 00:09:27,0 days 00:00:40,40.0,0 days 00:08:47,-2.75,2.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163,InsertingSourcePollutant,None,None,None,2.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20,0 days 00:07:27,0 days 00:01:35,95.0,0 days 00:05:52,-2.95,1.5,-1.85,3.5,-0.5,1.70,-2.25,0.40,2.282542,2.458150,1.303840
164,Insertin

In [50]:
userInputData.loc[:, ["x axis", "y axis"]]

,x axis,y axis
0,-2.35,3.1
1,-2.35,3.1
2,-2.35,3.6
3,-1.45,3.1
4,-1.95,2.6
...,...,...
161,NaN,NaN
162,-2.75,2.2
163,-2.95,1.5
164,-2.95,2.5


In [51]:
from matplotlib.ticker import FuncFormatter

def createTheTitleForTheFigure(experiment_number):
    date = userInputData.iloc[experiment_number]["timestamp InsertingSource"].date()
    row  = userInputData.iloc[experiment_number]
    if ((experiment_number > 0 )):
        previous_date = userInputData.iloc[experiment_number - 1]["timestamp InsertingSource"].date()
        if (date != previous_date):
            print("........................................................\n\n")
    
    title = "Experiment at row " + str(experiment_number)+ "At "+date.strftime("%Y-%m-%d")+ f" BME680:breathVocEquivalent.\n"
    if (userInputData.iloc[experiment_number]["experimentState"] == "InsertingSourcePollutant"):
        title= title + "at the pollution source position to be "
        if (pd.isna(userInputData.iloc[experiment_number]['front-wall'])==False):
           title =title + f"{userInputData.iloc[experiment_number]['front-wall']} meters from the front wall, "
        if (pd.isna(userInputData.iloc[experiment_number]['back-wall'])==False):
           title =title +f"{userInputData.iloc[experiment_number]['back-wall']} meters from the back wall, "
        if (pd.isna(userInputData.iloc[experiment_number]['side-right-wall'])==False):
           title =title+ f"{userInputData.iloc[experiment_number]['side-right-wall']} meters from the side right wall, "
        if (pd.isna(userInputData.iloc[experiment_number]['side-left-wall'])==False):
           title =title+ f"{userInputData.iloc[experiment_number]['side-left-wall']} meters from the side left wall, "
    elif (userInputData.iloc[experiment_number]["experimentState"] == "NoSourcePollutantInserted"):  
        title =title + "without source insertion.\n"
        
        
    
    title =title +"at the room:"+userInputData.iloc[experiment_number]["room"]+"\n"   
    title=title +"Στοιχεία για το μενωμένο πείραμα:"
    title= title+"Αντικείμενο που χρησιμοποιείται¨"+row["item-used"]+", "
    if (pd.isna(row["are-doors-opened"]) == False):
        title = title + "Οι πόρτες είναι ανοιχτές, "
    if (pd.isna(row["are-fans-on"]) == False):
        title = title + "Οι ανεμιστήρες είναι ενεργοποιημένοι, "
    if (pd.isna(row["are-people-inside"]) == False):
        title = title + "Βρίσκονται άνθρωποι μέσα, "
    if (pd.isna(row["are-windows-opened"]) == False):
        title = title + "Τα παράθυρα είναι ανοιχτά, "
    if (pd.isna(row["notes"])== False):
        title= title + "Σημειώσεις:"+ row["notes"]

    return title

def printData(dict_of_timeseries,plot_old_data = True):
    for experiment_number in dict_of_timeseries:
        particular_experiment =  dict_of_timeseries[experiment_number]
        
        title = createTheTitleForTheFigure(experiment_number)
            
       
        plt.figure(figsize=(18, 8))
        if plot_old_data:
            y_column_data = "VOC"
        else:
            y_column_data = "cutted_VOC"
            
        sns.lineplot(data=particular_experiment, x="seconds", y=y_column_data, hue="sensors")
        def format_seconds(x, _):
            return str(pd.to_timedelta(x, unit="s")).split()[-1]  # hh:mm:ss only
        plt.gca().xaxis.set_major_formatter(FuncFormatter(format_seconds))
        plt.xticks(rotation=45)
        
        plt.axvline(x=userInputData.at[experiment_number,"timestamp InsertingSource seconds"], color="red", linestyle="--", linewidth=2, label="timestamp of the inserting source")
        plt.title(title)
    
        plt.show()
    
        print("\n")

In [52]:
#printData(dict_of_timeseries,plot_old_data = True)

In [53]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,x axis,y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis,Euclidian distance to Id=0:BME680:breathVocEquivalent,Euclidian distance to Id=1:BME680:breathVocEquivalent,Euclidian distance to Id=2:BME680:breathVocEquivalent
0,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543.0,0 days 00:05:00,-2.35,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.750263,4.050309
1,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000.0,0 days 00:05:20,-2.35,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.750263,4.050309
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248.0,0 days 00:10:58,-2.35,3.6,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.781736,4.087175
3,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88.0,0 days 00:06:02,-1.45,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,3.850325,3.150397
4,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.3,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05,0 days 00:17:36,1056.0,0 days 00:08:29,-1.95,2.6,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.373214,3.677635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47,0 days 00:08:27,0 days 00:00:54,54.0,0 days 00:07:33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,InsertingSourcePollutant,None,None,None,1.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.5,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44,0 days 00:09:27,0 days 00:00:40,40.0,0 days 00:08:47,-2.75,2.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163,InsertingSourcePollutant,None,None,None,2.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20,0 days 00:07:27,0 days 00:01:35,95.0,0 days 00:05:52,-2.95,1.5,-1.85,3.5,-0.5,1.70,-2.25,0.40,2.282542,2.458150,1.303840
164,Insertin

In [54]:
dict_of_timeseries[6]

,sensors,VOC,after_insertion,original_value,datetime_timestamp,timestamp,seconds,VOC rolling average,VOC gradient,VOC gradient cumsum,VOC standard scaled,VOC rolling average standard scaled
0,Id=1:BME680:breathVocEquivalent,0.500000,False,True,2025-07-14 18:30:00,0 days 00:00:00,0.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
1,Id=2:BME680:breathVocEquivalent,0.500000,False,True,2025-07-14 18:30:00,0 days 00:00:00,0.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
2,Id=1:BME680:breathVocEquivalent,0.500000,False,False,2025-07-14 18:30:01,0 days 00:00:01,1.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
3,Id=2:BME680:breathVocEquivalent,0.500000,False,False,2025-07-14 18:30:01,0 days 00:00:01,1.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
4,Id=1:BME680:breathVocEquivalent,0.500000,False,False,2025-07-14 18:30:02,0 days 00:00:02,2.0,0.500000,0.000000,0.000000,-0.783998,-0.786811
...,...,...,...,...,...,...,...,...,...,...,...,...
6009,Id=2:BME680:breathVocEquivalent,18.846076,False,False,2025-07-14 19:20:01,0 days 00:50:01,3001.0,19.710940,-0.070793,19.176005,-0.745307,-0.746149
6010,Id=1:BME680:breathVocEquivalent,45.835785,False,False,2025-07-14 19:20:02,0 days 00:50:02,3002.0,48.097630,-0.234154,47.481665,-0.688386,-0.686067
6011,Id=2:BME680:breathVocEquivalent,18.729567,False,False,2025-07-14 19:20:02,0 days 00:50:02,3002.0,19.641070,-0.068507,19.107498,-0.745552,-0.746297
6012,Id=1:BME680:breathVocEquivalent,45.545510,False,True,2025-07-14 19:20:03,0 days 00:50:03,3003.0,47.865699,-0.231930,47.249734,-0.688998,-0.686557


In [55]:
#create a huge dataframe to save it 

# Combine into one DataFrame, keeping the dictionary key
timeSeriesData_BIG = pd.concat(dict_of_timeseries, names=["keys"])

# Reset index so the key becomes a column
timeSeriesData_BIG = timeSeriesData_BIG.reset_index(level="keys").reset_index(drop=True)
timeSeriesData_BIG["seconds"] = timeSeriesData_BIG["seconds"].astype(int)
timeSeriesData_BIG = timeSeriesData_BIG.set_index("seconds",drop=False)

In [56]:
timeSeriesData_BIG

,keys,sensors,VOC,after_insertion,original_value,datetime_timestamp,timestamp,seconds,VOC rolling average,VOC gradient,VOC gradient cumsum,VOC standard scaled,VOC rolling average standard scaled
seconds,,,,,,,,,,,,,
0,0,Id=1:BME680:breathVocEquivalent,1.472127,False,False,2025-07-01 15:27:19,0 days 00:00:00,0,1.477888,0.000436,0.000436,-0.777880,-0.775347
0,0,Id=2:BME680:breathVocEquivalent,1.430472,False,True,2025-07-01 15:27:19,0 days 00:00:00,0,1.460728,0.002113,0.002113,-0.798329,-0.783781
1,0,Id=1:BME680:breathVocEquivalent,1.468931,False,False,2025-07-01 15:27:20,0 days 00:00:01,1,1.478324,0.000326,0.000762,-0.779448,-0.775133
1,0,Id=2:BME680:breathVocEquivalent,1.434283,False,False,2025-07-01 15:27:20,0 days 00:00:01,1,1.462841,0.002084,0.004197,-0.796458,-0.782742
2,0,Id=1:BME680:breathVocEquivalent,1.469428,False,True,2025-07-01 15:27:21,0 days 00:00:02,2,1.478541,0.000201,0.000963,-0.779204,-0.775026
...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,165,Id=1:BME680:breathVocEquivalent,1.138193,True,True,2025-10-14 16:27:17,0 days 00:11:48,708,1.146781,-0.000690,0.330494,0.006772,0.020727
708,165,Id=0:BME680:breathVocEquivalent,1.141398,True,True,2025-10-14 16:27:17,0 days 00:11:48,708,1.166462,-0.002928,0.455692,0.010803,0.049516
709,165,Id=2:BME680:breathVocEquivalent,1.020477,True,True,2025-10-14 16:27:18,0 days 00:11:49,709,1.019839,-0.000731,0.348724,-0.141278,-0.164958


In [57]:
userInputData["timestamp InsertingSource timedelta"] = userInputData["timestamp InsertingSource timedelta"].astype("timedelta64[ns]")

In [58]:
userInputData.dtypes

experimentState                                                   object
are-doors-opened                                                  object
are-people-inside                                                 object
are-windows-opened                                                object
front-wall                                                       float64
item-used                                                         object
notes                                                             object
room                                                              object
side-right-wall                                                  float64
back-wall                                                        float64
side-left-wall                                                   float64
are-fans-on                                                       object
no-source-located                                                 object
timestamp InsertingSource                          

In [59]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,x axis,y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis,Euclidian distance to Id=0:BME680:breathVocEquivalent,Euclidian distance to Id=1:BME680:breathVocEquivalent,Euclidian distance to Id=2:BME680:breathVocEquivalent
0,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543.0,0 days 00:05:00,-2.35,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.750263,4.050309
1,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000.0,0 days 00:05:20,-2.35,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.750263,4.050309
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248.0,0 days 00:10:58,-2.35,3.6,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.781736,4.087175
3,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88.0,0 days 00:06:02,-1.45,3.1,NaN,NaN,2.4,3.05,1.70,3.05,NaN,3.850325,3.150397
4,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.3,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05,0 days 00:17:36,1056.0,0 days 00:08:29,-1.95,2.6,NaN,NaN,2.4,3.05,1.70,3.05,NaN,4.373214,3.677635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,NoSourcePollutantInserted,None,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,NaN,None,on,2025-10-14 14:56:13,2025-10-14 14:55:18,2025-10-14 15:03:49,2025-10-14,2025-10-14 14:55:20,2025-10-14 15:03:47,0 days 00:08:27,0 days 00:00:54,54.0,0 days 00:07:33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,InsertingSourcePollutant,None,None,None,1.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 ,...",NaN,NaN,0.5,None,None,2025-10-14 15:04:56,2025-10-14 15:04:15,2025-10-14 15:13:46,2025-10-14,2025-10-14 15:04:17,2025-10-14 15:13:44,0 days 00:09:27,0 days 00:00:40,40.0,0 days 00:08:47,-2.75,2.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163,InsertingSourcePollutant,None,None,None,2.5,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0...",NaN,NaN,0.3,None,None,2025-10-14 15:32:27,2025-10-14 15:30:51,2025-10-14 15:38:21,2025-10-14,2025-10-14 15:30:53,2025-10-14 15:38:20,0 days 00:07:27,0 days 00:01:35,95.0,0 days 00:05:52,-2.95,1.5,-1.85,3.5,-0.5,1.70,-2.25,0.40,2.282542,2.458150,1.303840
164,Insertin

In [60]:
timeSeriesData_BIG

,keys,sensors,VOC,after_insertion,original_value,datetime_timestamp,timestamp,seconds,VOC rolling average,VOC gradient,VOC gradient cumsum,VOC standard scaled,VOC rolling average standard scaled
seconds,,,,,,,,,,,,,
0,0,Id=1:BME680:breathVocEquivalent,1.472127,False,False,2025-07-01 15:27:19,0 days 00:00:00,0,1.477888,0.000436,0.000436,-0.777880,-0.775347
0,0,Id=2:BME680:breathVocEquivalent,1.430472,False,True,2025-07-01 15:27:19,0 days 00:00:00,0,1.460728,0.002113,0.002113,-0.798329,-0.783781
1,0,Id=1:BME680:breathVocEquivalent,1.468931,False,False,2025-07-01 15:27:20,0 days 00:00:01,1,1.478324,0.000326,0.000762,-0.779448,-0.775133
1,0,Id=2:BME680:breathVocEquivalent,1.434283,False,False,2025-07-01 15:27:20,0 days 00:00:01,1,1.462841,0.002084,0.004197,-0.796458,-0.782742
2,0,Id=1:BME680:breathVocEquivalent,1.469428,False,True,2025-07-01 15:27:21,0 days 00:00:02,2,1.478541,0.000201,0.000963,-0.779204,-0.775026
...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,165,Id=1:BME680:breathVocEquivalent,1.138193,True,True,2025-10-14 16:27:17,0 days 00:11:48,708,1.146781,-0.000690,0.330494,0.006772,0.020727
708,165,Id=0:BME680:breathVocEquivalent,1.141398,True,True,2025-10-14 16:27:17,0 days 00:11:48,708,1.166462,-0.002928,0.455692,0.010803,0.049516
709,165,Id=2:BME680:breathVocEquivalent,1.020477,True,True,2025-10-14 16:27:18,0 days 00:11:49,709,1.019839,-0.000731,0.348724,-0.141278,-0.164958


In [ ]:

sensors = timeSeriesData_BIG["sensors"].unique()

for sensor in sensors:
    col_name_before = f"{sensor} before insertion median"
    col_name_after = f"{sensor} after insertion median"
    medians_before = {}
    medians_after = {}
    medians_before_nat_log = {}
    medians_after_nat_log = {}  

    for key, row in userInputData.iterrows():
        # Get insertion time for this experiment
        insertion_time = row["timestamp InsertingSource seconds"]
        # Subset data for this sensor + experiment
        subset = timeSeriesData_BIG[
            (timeSeriesData_BIG["sensors"] == sensor) &
            (timeSeriesData_BIG["keys"] == key) 
        ]
        # Compute median (NaN if no data)
        medians_before[key] = subset.loc[subset.index <insertion_time]["VOC rolling average"].median()
        medians_after[key] = subset.loc[insertion_time>=subset.index ]["VOC rolling average"].median()
        medians_before_nat_log[key] = subset.loc[subset.index <insertion_time]["VOC rolling average standard scaled"].median()
        medians_after_nat_log[key] = subset.loc[insertion_time>=subset.index ]["VOC rolling average standard scaled"].median() 



    # Assign medians as a new column in userInputData
    userInputData[col_name_before] = pd.Series(medians_before)
    userInputData[col_name_after] = pd.Series(medians_after)
    
 
    userInputData[col_name_before+"VOC standard scaled"] = pd.Series(medians_before_nat_log)
    userInputData[col_name_after+"VOC standard scaled"] = pd.Series(medians_after_nat_log)


In [ ]:
userInputData.columns

In [ ]:
timeSeriesData_BIG

In [ ]:
def saveDataIntoDataFolder(data,data_file_name):
    script_dir = Path().resolve().parent
    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (data_file_name + ".json")
    with open(file_path, 'w') as file:
        if isinstance(data, pd.DataFrame):
            print("It's a DataFrame")
            if  data.empty:
                print("No data to save.")
            else:
                data.to_json(file_path, orient='records', lines=False)             
                print(f"Data saved to {file_path}")

        else:  
            print("It's NOT a DataFrame.")    
            if not data:
                print("No data to save.")
            else:    
                json.dump(data, file)
                print(f"Data saved to {file_path}")


saveDataIntoDataFolder(userInputData,"UserPrevious experiments-preprocessed")
saveDataIntoDataFolder(timeSeriesData_BIG,"Data:Previous experiments-preprocessed")